In [10]:
import yfinance as yf
import pandas as pd
import plotly.express as px

In [11]:
aapl= yf.download("AAPL", period="1y",auto_adjust=True)
aapl.head()

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,
2025-07-15,208.283691,211.052705,208.094440,208.393257,42296300
2025-07-16,209.329559,211.560698,207.815561,209.469006,47490500
2025-07-17,209.190109,210.963074,208.761800,209.737939,48068100
2025-07-18,210.345505,210.953095,208.871357,210.036732,48974600
2025-07-21,211.640381,214.927344,210.793749,211.261893,51377400


In [12]:
tsla = yf.download("TSLA", period="1y", auto_adjust=True)
tsla.tail()

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,TSLA,TSLA,TSLA,TSLA,TSLA
Date,,,,,
2026-07-08,394.059998,399.630005,390.510010,399.380005,33844900
2026-07-09,406.549988,407.859985,390.859985,393.989990,37835000
2026-07-10,407.760010,413.160004,402.809998,410.489990,33410000
2026-07-13,394.760010,405.570007,391.369995,404.609985,32809200
2026-07-14,NaN,NaN,NaN,NaN,23374603


In [13]:
aapl.columns = aapl.columns.get_level_values(0)
tsla.columns = tsla.columns.get_level_values(0)
aapl.head()

Price,Close,High,Low,Open,Volume
Date,,,,,
2025-07-15,208.283691,211.052705,208.094440,208.393257,42296300
2025-07-16,209.329559,211.560698,207.815561,209.469006,47490500
2025-07-17,209.190109,210.963074,208.761800,209.737939,48068100
2025-07-18,210.345505,210.953095,208.871357,210.036732,48974600
2025-07-21,211.640381,214.927344,210.793749,211.261893,51377400


In [14]:
print("Columns:", list(aapl.columns))
print("Index type:", type(aapl.index))
print("AAPL rows:", len(aapl))
print("TSLA rows:", len(tsla))
print("Date range:", aapl.index.min(), "→", aapl.index.max())

Columns: ['Close', 'High', 'Low', 'Open', 'Volume']
Index type: <class 'pandas.DatetimeIndex'>
AAPL rows: 251
TSLA rows: 251
Date range: 2025-07-15 00:00:00 → 2026-07-14 00:00:00


In [15]:
fig = px.line(aapl, y="Close", title="AAPL — Closing Price (1 Year)")
fig.show()

In [16]:
fig = px.line(tsla, y="Close", title="TSLA — Closing Price (1 Year)")
fig.show()

In [17]:
calendar_days= (aapl.index.max() - aapl.index.min()).days
print("Calendar days:", calendar_days)
print("Trading days:", len(aapl))
print("Missing days:", calendar_days - len(aapl))

Calendar days: 364
Trading days: 251
Missing days: 113


Alright, paper counts — here's the answer key. Grade yourself honestly: for each point, mark whether your version *actually said it*, not "I sort of meant that."

## Answer 1 — Why ~252 rows, not 365?

Stock data has one row per **trading day**, not per calendar day. Exchanges are closed on **weekends** (~104 days/year) and on **market holidays** like Christmas and Independence Day (~9–10 days/year for the NYSE). That leaves 365 − 104 − 10 ≈ **251–252 trading days** — which matches your data exactly: 364 calendar days spanned, 251 rows, 113 missing.


## Answer 2 — Why adjusted close?

In a stock split, a company multiplies its share count and the price divides accordingly — in a 3-for-1 split, a $300 stock becomes $100 overnight. **No value was lost**: every holder now owns 3× the shares. But the *raw* price chart shows a fake −66% crash. A model trained on raw prices would learn from a catastrophic event that never happened.

`auto_adjust=True` fixes this by **rescaling all historical prices** so the series is continuous, as if today's share count had always existed. Dividends are adjusted for the same reason: on the ex-dividend date the price drops by roughly the dividend amount, but holders received that money as cash — adjusted prices back the drop out, so the series reflects **true total return**, not an artificial dip.

